# PC diagnostic lab — Colab / GPU driver

Connect a Colab GPU kernel first: **Select Kernel → Colab → sign in → New Colab Server → GPU**.

This notebook does **not** rewrite the experiments as notebooks. It clones the repo onto the runtime (Colab cannot see your laptop disk) and shells out to the existing `.py` scripts.

C2 / C7 are fanned out with `scripts/run_parallel.py` (one process per cell×seed or λ×k×seed). That is what makes them hours instead of a week.

On one GPU keep `WORKERS = 2`. The settle loop is CPU-bound single-threaded Python, so throughput follows the number of CPU cores, not the GPU. One full-scale job is several hours and the full C2 grid is 27 of them, so it will not fit in one 12h session.

`PERSIST_TO_DRIVE` keeps `results/` on Google Drive, and C2 / C7 run with `--resume` into a fixed run dir. When a session dies, reconnect and run all cells again: finished jobs are skipped, and the merge only happens once every job is present.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/PandaGrdn/research-reimplementations.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    dest = Path("/content/research-reimplementations")
    if not dest.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(dest)])
    LAB = dest / "predictive_coding_proj" / "pc-diagnostic-lab"
else:
    here = Path.cwd().resolve()
    LAB = here if (here / "experiments").exists() else here / "predictive_coding_proj" / "pc-diagnostic-lab"

os.chdir(LAB)
sys.path.insert(0, str(LAB))
print("lab", LAB, "colab", IN_COLAB)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

WORKERS = 2 if DEVICE in {"cuda", "mps"} else min(8, os.cpu_count() or 4)
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "workers", WORKERS)

RUN_C11 = True
RUN_C13 = True
RUN_C0 = True
RUN_C4 = True
RUN_C2 = True
RUN_C7 = True
RUN_FIGURES = True
SEEDS = ["0", "1", "2"]
OVERLAY = ["configs/reporting.yaml"]
C2_CELLS = None
C7_KS = None
C7_LAMBDAS = None

# Keep results/ on Drive so a dropped session loses only the jobs that were mid-run.
PERSIST_TO_DRIVE = True
if IN_COLAB and PERSIST_TO_DRIVE:
    import shutil
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        keep = Path("/content/drive/MyDrive/pc_lab_results")
        keep.mkdir(parents=True, exist_ok=True)
        res = LAB / "results"
        if not res.is_symlink():
            shutil.copytree(res, keep, dirs_exist_ok=True)
            shutil.rmtree(res)
            res.symlink_to(keep, target_is_directory=True)
        print("results ->", keep)
    except Exception as e:
        print("Drive not mounted; results stay on the session disk:", e)


In [ ]:
def run(cmd):
    print(">>>", " ".join(cmd), flush=True)
    subprocess.check_call(cmd)

common = ["--device", DEVICE, "--overlay", *OVERLAY]
seed_args = ["--seeds", *SEEDS]

if RUN_C11:
    run([sys.executable, "experiments/c11_offline_gru.py", *seed_args, *common])
if RUN_C13:
    run([sys.executable, "experiments/c13_consistency_rollout.py", *seed_args, *common])
if RUN_C0:
    run([sys.executable, "experiments/c0_isolation_test.py", *seed_args, "--device", DEVICE])
if RUN_C4:
    run([sys.executable, "experiments/c4_settle_determinism.py", *seed_args, "--device", DEVICE])

In [ ]:
if RUN_C2:
    cmd = [sys.executable, "scripts/run_parallel.py", "c2", "--workers", str(WORKERS), "--device", DEVICE, *seed_args,
           "--overlay", *OVERLAY, "--resume", "results/c2_mitigation_grid/colab"]
    if C2_CELLS:
        cmd += ["--cells", *C2_CELLS]
    run(cmd)
if RUN_C7:
    cmd = [sys.executable, "scripts/run_parallel.py", "c7", "--workers", str(WORKERS), "--device", DEVICE, *seed_args,
           "--overlay", *OVERLAY, "--resume", "results/c7_slowness_sweep/colab"]
    if C7_KS:
        cmd += ["--unroll-k", *[str(k) for k in C7_KS]]
    if C7_LAMBDAS:
        cmd += ["--lambda-slow", *[str(x) for x in C7_LAMBDAS]]
    run(cmd)
if RUN_FIGURES:
    run([sys.executable, "scripts/make_figures.py"])
print("done. results in", LAB / "results")

In [ ]:
if IN_COLAB:
    zip_path = Path("/content/pc_lab_results.zip")
    subprocess.check_call(["zip", "-r", str(zip_path), "results", "figures", "artifacts"], cwd=str(LAB))
    from google.colab import files
    files.download(str(zip_path))